# Supplementary Data Table 5

Environment: env1

In [ ]:
import pandas as pd
import pickle
import os
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, roc_curve, auc, precision_recall_curve
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix
from scipy.stats import sem
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
import sys
sys.path.append("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/src/analysis/final_figures/final_submission/")
from utils import classification_metrics



In [3]:
all_mets = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_strat_pec_hc_pw/run_20260407_stable/meta_probs_auc_0.5/all_pw_metrics.csv")
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")
result = all_mets.loc[all_mets.groupby('pw')['auc'].idxmax(), ['pw', 'hp', 'auc']]
result.head()

,pw,hp,auc
0,1,"{n_estimators_250,max_depth_3,learning_rate_0....",0.760809
324,2,"{n_estimators_250,max_depth_2,learning_rate_0....",0.663553
650,3,"{n_estimators_100,max_depth_3,learning_rate_0....",0.899518
982,4,"{n_estimators_250,max_depth_3,learning_rate_0....",0.793789
1296,5,"{n_estimators_250,max_depth_1,learning_rate_0....",0.715025


In [4]:
cu_all_probs = []
for pw in range(1, 6):
    hp = result[result["pw"] == pw]["hp"].iloc[0]
    path = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_strat_pec_hc_pw{pw}/run_20260407_stable/meta_probs_auc_0.5/XGBoost/{hp}_oof.pkl"
    with open(path, "rb") as f:
        data = pickle.load(f)
        cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

In [ ]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left", on='id')

In [6]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_pec_20251017.pkl", "rb") as f:
    pec = pickle.load(f)
with open("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/rev_test_pec_cases_20260406.pkl", "rb") as f:
    pec_test = pickle.load(f)
all_pec = pec + pec_test
cu_all_probs["label"] = [int(id in all_pec) for id in cu_all_probs["id"]]

cu_nsf = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_nsf_20251028.pkl')
cu_sf = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_sf_20251028.pkl')
cu_eope = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_eope_20251017.pkl')
cu_lope = pd.read_pickle('/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_lope_20251028.pkl')

cases = cu_nsf+cu_sf+cu_eope+cu_lope
cases = list(set(cases))

rename_dict = {}

for i in range(6):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [7]:
cu_all_probs['avg_prob'] = cu_all_probs[['pw_1', 'pw_2', 'pw_3', 'pw_4', 'pw_5']].mean(axis=1)

In [8]:
reweight = cu_all_probs.copy()
cols = ["Combo"]
reweight["Combo"] = (reweight["Past PEC"] * 0.35 + 0.15 *((reweight["First Preg"]) * (reweight["Age at enrollment"] > 37).astype(int)))

In [9]:
cases_types = {'sf':cu_sf,'nsf':cu_nsf,'eope':cu_eope,'lope':cu_lope,'pec':cases}

for k,ct in cases_types.items():

    print(k)

    t = reweight[(reweight['id'].isin(ct))|(~reweight['id'].isin(cases))]
    cols = [f'pw_{i}' for i in range(1,6)]
    cols.append('Combo')
    t['pred'] = t[cols].mean(axis=1)

    reweight['pred'] = reweight[cols].mean(axis=1)

    eval_col = 'pred'

    fpr, tpr, thresholds = roc_curve(t['label'], t[eval_col])
    roc_df = pd.DataFrame({'fpr': fpr, 'tpr': tpr, 'threshold': thresholds})
    whole_fpr, whole_tpr, whole_thresholds = roc_curve(reweight['label'], reweight[eval_col])
    whole_roc_df = pd.DataFrame({'fpr': whole_fpr, 'tpr': whole_tpr, 'threshold': whole_thresholds})

    target=0.3

    row = whole_roc_df.iloc[(whole_roc_df['fpr'] - target).abs().argsort()[0]]
    thresh = row['threshold']
    
    preds = (t[eval_col] >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(t['label'], preds).ravel()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f"\nFPR ~{target:.0%} (threshold={thresh:.3f}):")
    print(f"  TPR/Recall: {recall:.3f}  |  Precision: {precision:.3f}")
    print(f"  TP={tp}, FP={fp}, TN={tn}, FN={fn}")
    
    cm = classification_metrics(tp, fn, fp, tn)
    for i in cm:
        print(f'{i}: {cm[i]}')

    t['label'].sum()

    classification_metrics(tp, fn, fp, tn)
    print()

sf

FPR ~30% (threshold=0.207):
  TPR/Recall: 0.974  |  Precision: 0.102
  TP=37, FP=326, TN=808, FN=1
TP: 37
FP: 326
TN: 808
FN: 1
TPR: 0.9736842105263158
FPR: 0.2874779541446208
PPV: 0.10192837465564739
NPV: 0.9987639060568603
F1: 0.18453865336658357

nsf

FPR ~30% (threshold=0.207):
  TPR/Recall: 0.938  |  Precision: 0.044
  TP=15, FP=326, TN=808, FN=1
TP: 15
FP: 326
TN: 808
FN: 1
TPR: 0.9375
FPR: 0.2874779541446208
PPV: 0.04398826979472141
NPV: 0.9987639060568603
F1: 0.08403361344537816

eope

FPR ~30% (threshold=0.207):
  TPR/Recall: 1.000  |  Precision: 0.058
  TP=20, FP=326, TN=808, FN=0
TP: 20
FP: 326
TN: 808
FN: 0
TPR: 1.0
FPR: 0.2874779541446208
PPV: 0.057803468208092484
NPV: 1.0
F1: 0.10928961748633878

lope

FPR ~30% (threshold=0.207):
  TPR/Recall: 0.941  |  Precision: 0.089
  TP=32, FP=326, TN=808, FN=2
TP: 32
FP: 326
TN: 808
FN: 2
TPR: 0.9411764705882353
FPR: 0.2874779541446208
PPV: 0.0893854748603352
NPV: 0.9975308641975309
F1: 0.163265306122449

pec

FPR ~30% (threshol

/gpfs/data/shenhavlab/users/Daniel/.conda/envs/env1/lib/python3.7/site-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.
/gpfs/data/shenhavlab/users/Daniel/.conda/envs/env1/lib/python3.7/site-packages/ipykernel_launcher.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # Remove the CWD from sys.path while we load stuff.
/gpfs/data/shenhavlab/users/Daniel/.conda/envs/env1/lib/python3.7/site-packages/ipykernel_launcher.py:10